In [3]:
import torch

inputs = torch.tensor([[0.72, 0.45, 0.31], # Dream 
                    [0.75, 0.20, 0.55], # big 
                    [0.30, 0.80, 0.40], # and 
                    [0.85, 0.35, 0.60], # work 
                    [0.55, 0.15, 0.75], # for 
                    [0.25, 0.20, 0.85]]) # it 
words = ['Dream','big','and','work','for','it']

In [20]:
import torch.nn as nn
import math
class CasualAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Parameter(torch.rand(d_in,d_out,bias = qkv_bias))
        self.W_key = nn.Parameter(torch.rand(d_in,d_out,bias = qkv_bias))
        self.W_value = nn.Parameter(torch.rand(d_in,d_out,bias = qkv_bias))
        self.dropout = dropout
        self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        Q = x.W_query
        K = x.W_key
        V = x.W_value
        b,num_token,d_in = x.shape
        attention_scores = Q * K.T
        attention_scores.masked_fill(self.mask.bool()[:num_token,:num_token],-torch.inf)
        attention_weights = torch.softmax(attention_scores/sqrt(K.shape[-1]),dim=-1)
        attention_weights = self.dropout(attention_weights)
        context_vector = attention_weights*V
        return context_vector

In [21]:
d_in = inputs.shape[-1]
d_out = 2

In [22]:
batch = torch.stack((inputs,inputs,inputs,inputs),dim=0)
print(batch.shape)

torch.Size([4, 6, 3])


In [29]:
import torch.nn as nn
import math
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,num_heads,qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CasualAttention(d_in,d_out,context_length,dropout,qkv_bias)]
            for _ in range(num_heads)
        )
    def forward(self,x):
        return torch.cat([head(x) for head in self.heads],dim=-1)

In [30]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in,d_out = 3,2
mha = MultiHeadAttentionWrapper(d_in,d_out,context_length, 0.0,num_heads=4)

TypeError: rand() received an invalid combination of arguments - got (int, int, bias=bool), but expected one of:
 * (tuple of ints size, *, torch.Generator generator, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)


In [15]:
context_vectors = mha(batch)
print(context_vectors)

NameError: name 'mha' is not defined